# Notebook 3 — Modelli Avanzati, Analisi Causale e Previsioni LA 2028

---

Notebook conclusivo. Costruiamo modelli predittivi, testiamo la causalità
con il Difference-in-Differences, clusterizziamo i paesi e prevediamo
le medaglie a Los Angeles 2028.

Tutti i grafici con **Altair** — interattivi.


## 1. Setup e preparazione dati

In [29]:
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats
from sklearn.linear_model import LinearRegression, LassoCV, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

NAVY=  "#0A2342"; BLUE=  "#1A6B9A"; TEAL=  "#0D9488"
GOLD=  "#F59E0B"; RED=   "#DC2626"; GREEN= "#16A34A"; GRAY= "#64748B"

df = pd.read_csv("olympic_medals_clean.csv")
df_summer = df[df["is_winter"]==0].copy()
df_summer["log_gdp_total"] = np.log1p(df_summer["NY.GDP.MKTP.CD"])
df_summer["log_pop"]       = np.log1p(df_summer["SP.POP.TOTL"])
df_summer["log_gdp_pc"]    = np.log1p(df_summer["NY.GDP.PCAP.CD"])
df_summer["medals_per_million"] = np.where(
    df_summer["SP.POP.TOTL"]>0,
    df_summer["total"]/(df_summer["SP.POP.TOTL"]/1e6), np.nan)

host_map = {1964:"Japan",1968:"Mexico",1972:"West Germany",1976:"Canada",
            1980:"Soviet Union",1984:"United States",1988:"South Korea",
            1992:"Spain",1996:"United States",2000:"Australia",
            2004:"Greece",2008:"China",2012:"Great Britain",
            2016:"Brazil",2020:"Japan"}
df_summer["host"] = df_summer.apply(
    lambda r: 1 if host_map.get(r["year"],"") == r["country"] else 0, axis=1)
df_summer = df_summer.sort_values(["country","year"])
df_summer["prev_medals"]   = df_summer.groupby("country")["total"].shift(1)
df_summer["prev_medals_2"] = df_summer.groupby("country")["total"].shift(2)

print("Setup ✅")


Setup ✅


## 2. Imputazione dati mancanti

In [2]:
# Interpolazione lineare entro paese per riempire i buchi delle serie storiche
impute_cols = ["NY.GDP.MKTP.CD","NY.GDP.PCAP.CD","SP.POP.TOTL",
               "SP.DYN.LE00.IN","SP.URB.TOTL.IN.ZS","EN.POP.DNST"]

df_imp = df_summer.copy()
before = {c: df_imp[c].isna().sum() for c in impute_cols}
for col in impute_cols:
    df_imp[col] = (df_imp.groupby("country")[col]
                   .transform(lambda s: s.interpolate(
                       method="linear", limit_direction="both", limit_area="inside")))
after  = {c: df_imp[c].isna().sum() for c in impute_cols}
df_imp["log_gdp_total"] = np.log1p(df_imp["NY.GDP.MKTP.CD"])
df_imp["log_pop"]       = np.log1p(df_imp["SP.POP.TOTL"])
df_imp["log_gdp_pc"]    = np.log1p(df_imp["NY.GDP.PCAP.CD"])

# Grafico impatto imputazione
imp_df = pd.DataFrame({
    "variabile": impute_cols * 2,
    "valore": [before[c] for c in impute_cols] + [after[c] for c in impute_cols],
    "momento": ["Prima"]*len(impute_cols) + ["Dopo"]*len(impute_cols)
})

alt.Chart(imp_df).mark_bar().encode(
    x=alt.X("valore:Q", title="N. valori mancanti"),
    y=alt.Y("variabile:N", title=None),
    color=alt.Color("momento:N",
        scale=alt.Scale(domain=["Prima","Dopo"], range=[RED, TEAL]),
        legend=alt.Legend(title="")),
    xOffset="momento:N",
    tooltip=["variabile:N","momento:N",alt.Tooltip("valore:Q", title="NaN")]
).properties(title="Impatto dell'imputazione per variabile", width=480, height=240).display()


alt.Chart(...)

## 3. Modello predittivo — confronto 5 algoritmi

In [30]:
features = ["log_gdp_total","log_pop","log_gdp_pc",
            "SP.DYN.LE00.IN","SP.URB.TOTL.IN.ZS","host","prev_medals"]

model_data = df_imp.dropna(subset=features+["total"]).copy()
model_data  = model_data[model_data["year"] <= 2016]
X = model_data[features].values
y = model_data["total"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "OLS"             : Pipeline([("sc",StandardScaler()),("m",LinearRegression())]),
    "Ridge"           : Pipeline([("sc",StandardScaler()),("m",Ridge(alpha=1.0))]),
    "Lasso CV"        : Pipeline([("sc",StandardScaler()),("m",LassoCV(cv=5,max_iter=5000))]),
    "Random Forest"   : RandomForestRegressor(n_estimators=200,max_depth=8,
                                               min_samples_leaf=5,random_state=42),
    "Gradient Boosting":GradientBoostingRegressor(n_estimators=200,max_depth=4,
                                                   learning_rate=0.05,random_state=42),
}
cv_results = []
for name, model in models.items():
    r2s  = cross_val_score(model, X, y, cv=kf, scoring="r2")
    maes = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")
    cv_results.append({"modello":name,"R2":r2s.mean(),"R2_std":r2s.std(),"MAE":maes.mean()})

cv_df = pd.DataFrame(cv_results).sort_values("R2")

# Bar chart R² con error bars
bars_cv = (
    alt.Chart(cv_df)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("R2:Q", title="R² (cross-validation 5-fold)", scale=alt.Scale(domain=[0,1])),
        y=alt.Y("modello:N", sort="-x", title=None),
        color=alt.condition(
            alt.datum.modello == "Random Forest",
            alt.value(TEAL), alt.value(BLUE)),
        tooltip=["modello:N",
                 alt.Tooltip("R2:Q", format=".3f", title="R²"),
                 alt.Tooltip("R2_std:Q", format=".3f", title="Std"),
                 alt.Tooltip("MAE:Q", format=".2f", title="MAE")]
    )
)
errors_cv = (
    alt.Chart(cv_df)
    .mark_errorbar(color=NAVY)
    .encode(
        x=alt.X("R2:Q"),
        xError=alt.XError("R2_std:Q"),
        y="modello:N"
    )
)
text_cv = (
    alt.Chart(cv_df)
    .mark_text(dx=5, fontSize=10)
    .encode(x="R2:Q", y=alt.Y("modello:N", sort="-x"),
            text=alt.Text("R2:Q", format=".3f"))
)
(bars_cv + errors_cv + text_cv).properties(
    title="Confronto modelli — R² medio (cross-validation 5-fold)",
    width=520, height=240).display()


alt.LayerChart(...)

### 3.1 Feature importance — Random Forest

In [31]:
rf = RandomForestRegressor(n_estimators=300,max_depth=8,min_samples_leaf=5,random_state=42)
rf.fit(X, y)

imp_df = (pd.DataFrame({"feature":features,"importance":rf.feature_importances_})
          .sort_values("importance"))

labels_map = {
    "log_gdp_total":"log(PIL totale)","log_pop":"log(Popolazione)",
    "log_gdp_pc":"log(PIL pro capite)","SP.DYN.LE00.IN":"Aspett. vita",
    "SP.URB.TOTL.IN.ZS":"% Urbanizzazione","host":"Host (paese ospitante)",
    "prev_medals":"Medaglie ciclo precedente"
}
imp_df["label"] = imp_df["feature"].map(labels_map)

chart_imp = (
    alt.Chart(imp_df)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("importance:Q", title="Importanza relativa"),
        y=alt.Y("label:N", sort="-x", title=None),
        color=alt.condition(
            alt.datum.importance > 0.15,
            alt.value(RED), alt.value(BLUE)),
        tooltip=["label:N", alt.Tooltip("importance:Q", format=".3f", title="Importanza")]
    )
)
text_imp = (
    alt.Chart(imp_df)
    .mark_text(dx=5, fontSize=10)
    .encode(x="importance:Q", y=alt.Y("label:N", sort="-x"),
            text=alt.Text("importance:Q", format=".3f"))
)
(chart_imp + text_imp).properties(
    title="Random Forest — importanza delle variabili nel predire le medaglie",
    width=500, height=280).display()

print("prev_medals è la variabile più importante — il passato spiega il futuro.")
print("log_gdp_total è seconda — coerente con tutto il Notebook 1.")


alt.LayerChart(...)

prev_medals è la variabile più importante — il passato spiega il futuro.
log_gdp_total è seconda — coerente con tutto il Notebook 1.


### 3.2 Test out-of-sample — Tokyo 2020

In [32]:
# Allenato su 1968-2016, testato su Tokyo 2020 (dati mai visti)
test_data = df_imp[df_imp["year"]==2020].dropna(subset=features+["total"]).copy()
rf.fit(X, y)
test_data["previsto"]  = np.maximum(rf.predict(test_data[features].values), 0).round(1)
test_data["errore"]    = (test_data["previsto"] - test_data["total"]).round(1)
test_data["abs_err"]   = test_data["errore"].abs()

r2_t  = r2_score(test_data["total"], test_data["previsto"])
mae_t = mean_absolute_error(test_data["total"], test_data["previsto"])

# Scatter previsto vs reale
lim = max(test_data["total"].max(), test_data["previsto"].max()) * 1.05
diag = pd.DataFrame({"x":[0,lim], "y":[0,lim]})

scatter_test = (
    alt.Chart(test_data)
    .mark_circle(size=60, opacity=0.6, color=BLUE)
    .encode(
        x=alt.X("total:Q", title="Medaglie reali (Tokyo 2020)",
                scale=alt.Scale(domain=[0,lim])),
        y=alt.Y("previsto:Q", title="Medaglie previste",
                scale=alt.Scale(domain=[0,lim])),
        size=alt.Size("abs_err:Q", legend=None),
        tooltip=["country:N","country_noc:N",
                 alt.Tooltip("total:Q", title="Reale"),
                 alt.Tooltip("previsto:Q", title="Previsto"),
                 alt.Tooltip("errore:Q", title="Errore", format="+.1f")]
    )
)
labels_test = (
    alt.Chart(test_data[test_data["total"]>25])
    .mark_text(dx=8, dy=-5, fontSize=9)
    .encode(x="total:Q", y="previsto:Q", text="country_noc:N")
)
diag_line = (alt.Chart(diag).mark_line(strokeDash=[4,4], color=GRAY)
             .encode(x="x:Q", y="y:Q"))

(scatter_test + labels_test + diag_line).properties(
    title=f"Previsto vs reale Tokyo 2020 — R²={r2_t:.3f}  MAE={mae_t:.1f} medaglie",
    width=500, height=440
).interactive().display()


alt.LayerChart(...)

## 4. Analisi per disciplina — 3 nazioni

In [33]:
discipline_data = pd.DataFrame({
    "sport":["Athletics","Swimming","Cycling","Rowing","Sailing",
             "Gymnastics","Boxing","Judo","Canoeing"]*3,
    "country":["Great Britain"]*9+["France"]*9+["Australia"]*9,
    "avg_invest_M":[
        23.3,24.1,27.3,31.1,23.5,16.8,12.2,7.3,10.2,
        28.5,22.0,18.5,12.0,15.0,20.0,8.5,14.5,7.0,
        25.0,30.0,15.0,18.0,20.0,12.0,8.0,4.0,9.0,
    ],
    "avg_medals":[
        4.0,4.25,9.5,7.25,3.75,3.75,4.25,1.5,2.25,
        2.0,1.75,2.0,1.5,1.75,2.5,1.0,3.5,1.0,
        2.25,4.75,2.0,2.5,3.25,1.0,0.5,0.25,1.75,
    ],
})

scatter_disc = (
    alt.Chart(discipline_data)
    .mark_point(size=90, filled=True, opacity=0.85)
    .encode(
        x=alt.X("avg_invest_M:Q", title="Investimento medio per ciclo (M)"),
        y=alt.Y("avg_medals:Q", title="Medaglie medie per Olimpiade"),
        color=alt.Color("country:N",
            scale=alt.Scale(domain=["Great Britain","France","Australia"],
                            range=[BLUE, RED, TEAL])),
        tooltip=["sport:N","country:N",
                 alt.Tooltip("avg_invest_M:Q", title="Invest. medio", format=".1f"),
                 alt.Tooltip("avg_medals:Q", title="Medaglie medie", format=".2f")]
    )
)
labels_disc = (
    alt.Chart(discipline_data)
    .mark_text(dx=6, dy=-5, fontSize=8)
    .encode(x="avg_invest_M:Q", y="avg_medals:Q",
            text="sport:N",
            color=alt.Color("country:N",
                scale=alt.Scale(domain=["Great Britain","France","Australia"],
                                range=[BLUE, RED, TEAL])))
)
reg_disc = (
    alt.Chart(discipline_data)
    .transform_regression("avg_invest_M","avg_medals", groupby=["country"])
    .mark_line(strokeDash=[4,4])
    .encode(x="avg_invest_M:Q", y="avg_medals:Q",
            color="country:N")
)

(scatter_disc + labels_disc + reg_disc).properties(
    title="Investimento per disciplina vs medaglie — GB, Francia, Australia",
    width=600, height=380
).display()

for country, grp in discipline_data.groupby("country"):
    r, p = stats.pearsonr(grp["avg_invest_M"], grp["avg_medals"])
    print(f"{country:<18}: r={r:.3f}  p={p:.4f}")


alt.LayerChart(...)

Australia         : r=0.902  p=0.0009
France            : r=0.410  p=0.2732
Great Britain     : r=0.781  p=0.0130


## 5. Difference-in-Differences

In [34]:
# SPLISS DiD: Tokyo→Paris — chi ha aumentato la spesa ha guadagnato medaglie?
spliss_did = pd.DataFrame({
    "country_noc":["GBR","NED","FRA","GER","AUS","BEL","CAN","JPN","BRA","POL","NOR","DEN","SUI","KOR"],
    "country":["Great Britain","Netherlands","France","Germany","Australia","Belgium",
               "Canada","Japan","Brazil","Poland","Norway","Denmark","Switzerland","South Korea"],
    "spend_tokyo":[331,108,220,185,200,25,140,400,180,55,65,38,42,160],
    "medals_tokyo":[65,36,33,37,46,9,24,58,21,14,8,11,13,20],
    "spend_paris":[350,120,280,195,220,28,155,380,190,60,72,42,48,175],
    "medals_paris":[65,34,66,33,53,10,27,45,20,10,8,9,5,32],
})
spliss_did["delta_spend"]  = spliss_did["spend_paris"] - spliss_did["spend_tokyo"]
spliss_did["delta_medals"] = spliss_did["medals_paris"] - spliss_did["medals_tokyo"]
spliss_did["delta_pct"]    = spliss_did["delta_spend"] / spliss_did["spend_tokyo"] * 100
spliss_did["gruppo"] = pd.cut(spliss_did["delta_pct"],
    bins=[-np.inf,-15,15,40,np.inf],
    labels=["Taglio","Stabile","Aumento mod.","Aumento forte"])
spliss_did["direzione"] = spliss_did["delta_medals"].apply(
    lambda v: "Più medaglie" if v>0 else "Meno medaglie")

r_did, _ = stats.pearsonr(spliss_did["delta_spend"], spliss_did["delta_medals"])

# Bar medie per gruppo + scatter
grp_means = (spliss_did.groupby("gruppo",observed=True)["delta_medals"]
             .mean().reset_index().rename(columns={"delta_medals":"delta_medio"}))

bars_did = (
    alt.Chart(grp_means)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("gruppo:N", sort=["Taglio","Stabile","Aumento mod.","Aumento forte"],
                title="Gruppo di spesa"),
        y=alt.Y("delta_medio:Q", title="Δ medaglie medio"),
        color=alt.Color("gruppo:N",
            scale=alt.Scale(
                domain=["Taglio","Stabile","Aumento mod.","Aumento forte"],
                range=[RED,"#94A3B8","#60A5FA",BLUE])),
        tooltip=["gruppo:N", alt.Tooltip("delta_medio:Q", format="+.1f", title="Δ medio")]
    )
)
rule0 = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(color=NAVY).encode(y="y:Q")

scatter_did = (
    alt.Chart(spliss_did)
    .mark_point(size=90, filled=True, opacity=0.85)
    .encode(
        x=alt.X("delta_pct:Q", title="Δ% spesa (Tokyo→Paris)"),
        y=alt.Y("delta_medals:Q", title="Δ medaglie (Tokyo→Paris)"),
        color=alt.Color("direzione:N",
            scale=alt.Scale(domain=["Più medaglie","Meno medaglie"], range=[TEAL,RED])),
        tooltip=["country:N","country_noc:N",
                 alt.Tooltip("delta_pct:Q", format="+.1f", title="Δ% spesa"),
                 alt.Tooltip("delta_medals:Q", format="+.0f", title="Δ medaglie")]
    )
)
labels_did = (
    alt.Chart(spliss_did).mark_text(dx=7,dy=-5,fontSize=9)
    .encode(x="delta_pct:Q", y="delta_medals:Q", text="country_noc:N")
)
reg_did = (
    alt.Chart(spliss_did)
    .transform_regression("delta_pct","delta_medals")
    .mark_line(strokeDash=[4,4], color=NAVY)
    .encode(x="delta_pct:Q", y="delta_medals:Q")
)
hline_did = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(color=GRAY,strokeDash=[3,3]).encode(y="y:Q")
vline_did = alt.Chart(pd.DataFrame({"x":[0]})).mark_rule(color=GRAY,strokeDash=[3,3]).encode(x="x:Q")

alt.hconcat(
    (bars_did+rule0).properties(title="Δ medaglie per gruppo", width=280, height=280),
    (scatter_did+labels_did+reg_did+hline_did+vline_did).properties(
        title=f"Δ% spesa vs Δ medaglie — r={r_did:.2f}", width=340, height=280)
).display()


alt.HConcatChart(...)

## 6. Cluster di paesi — quattro profili olimpici

In [35]:
profile = (df_imp[df_imp["total"]>0].groupby("country").agg(
    avg_medals=("total","mean"), total_medals=("total","sum"),
    n_editions=("year","nunique"), avg_gdp_pc=("NY.GDP.PCAP.CD","mean"),
    avg_pop=("SP.POP.TOTL","mean"), medals_per_m=("medals_per_million","mean"),
    consistency=("total","std")).dropna().reset_index())

feats_cl = ["avg_medals","medals_per_m","consistency","avg_gdp_pc","n_editions"]
X_cl = StandardScaler().fit_transform(profile[feats_cl].fillna(0))
km   = KMeans(n_clusters=4, random_state=42, n_init=20)
profile["cluster"] = km.fit_predict(X_cl)

# Nome del cluster in base alle caratteristiche medie
c_names = {}
for c in range(4):
    s = profile[profile["cluster"]==c]
    c_names[c] = ("Potenze assolute"  if s["avg_medals"].mean()>40  else
                  "Specializzati"     if s["medals_per_m"].mean()>2  else
                  "Emergenti/Medi"    if s["avg_medals"].mean()>8    else
                  "Partecipanti")
profile["cluster_name"] = profile["cluster"].map(c_names)
profile["log_gdp_pc"]   = np.log1p(profile["avg_gdp_pc"])

# Paesi da evidenziare con etichetta
highlight = ["United States","China","Soviet Union","Jamaica","Kenya",
             "Great Britain","Italy","Germany","Australia","Cuba","Norway"]

scatter_cl = (
    alt.Chart(profile)
    .mark_circle(size=55, opacity=0.7)
    .encode(
        x=alt.X("log_gdp_pc:Q", title="log(PIL pro capite medio)"),
        y=alt.Y("avg_medals:Q", title="Medaglie medie per edizione"),
        color=alt.Color("cluster_name:N",
            scale=alt.Scale(
                domain=["Potenze assolute","Specializzati","Emergenti/Medi","Partecipanti"],
                range=[RED, TEAL, BLUE, GRAY]),
            legend=alt.Legend(title="Profilo")),
        tooltip=["country:N","cluster_name:N",
                 alt.Tooltip("avg_medals:Q", format=".1f", title="Med. medie"),
                 alt.Tooltip("medals_per_m:Q", format=".2f", title="Med/milione"),
                 alt.Tooltip("avg_gdp_pc:Q", format=",.0f", title="PIL p.c. medio")]
    )
)
labels_cl = (
    alt.Chart(profile[profile["country"].isin(highlight)])
    .mark_text(dx=7, dy=-5, fontSize=8.5, fontWeight="bold")
    .encode(x="log_gdp_pc:Q", y="avg_medals:Q", text="country:N",
            color=alt.Color("cluster_name:N",
                scale=alt.Scale(
                    domain=["Potenze assolute","Specializzati","Emergenti/Medi","Partecipanti"],
                    range=[RED, TEAL, BLUE, GRAY])))
)

(scatter_cl + labels_cl).properties(
    title="Quattro profili olimpici — clicca su un punto per i dettagli",
    width=620, height=420
).interactive().display()


alt.LayerChart(...)

## 7. Previsioni Los Angeles 2028

In [36]:
paris_2024 = {
    "United States":126,"Great Britain":65,"China":91,"Australia":53,
    "France":66,"Netherlands":34,"Germany":33,"Japan":45,"Italy":40,
    "Canada":27,"New Zealand":20,"South Korea":32,"Brazil":20,
    "Kenya":11,"Jamaica":9,"Cuba":6,"Norway":8,"Sweden":6,
    "Hungary":6,"Spain":5,"Poland":10,"Greece":4,"Ethiopia":5,
}

pred_rows = []
for country, prev_med in paris_2024.items():
    hist = df_imp[df_imp["country"]==country].sort_values("year")
    if hist.empty: continue
    last = hist.iloc[-1]
    recent = hist.tail(3)["NY.GDP.MKTP.CD"].dropna()
    g = np.clip(recent.pct_change().mean() if len(recent)>1 else 0.025, -0.05, 0.10)
    proj_gdp = last["NY.GDP.MKTP.CD"]*(1+g)**2
    proj_pop = last["SP.POP.TOTL"]*1.015
    pred_rows.append({
        "country":country,
        "log_gdp_total":np.log1p(proj_gdp),
        "log_pop":np.log1p(proj_pop),
        "log_gdp_pc":np.log1p(proj_gdp/proj_pop if proj_pop>0 else 0),
        "SP.DYN.LE00.IN":last["SP.DYN.LE00.IN"]+0.5,
        "SP.URB.TOTL.IN.ZS":min(last["SP.URB.TOTL.IN.ZS"]+0.5,100),
        "host":1 if country=="United States" else 0,
        "prev_medals":prev_med,
    })

pred_df = pd.DataFrame(pred_rows).dropna()
features = ["log_gdp_total","log_pop","log_gdp_pc","SP.DYN.LE00.IN","SP.URB.TOTL.IN.ZS","host","prev_medals"]

rf_f = RandomForestRegressor(n_estimators=300,max_depth=8,min_samples_leaf=5,random_state=42)
model_data = df_imp.dropna(subset=features+["total"]).copy()
model_data  = model_data[model_data["year"]<=2016]
rf_f.fit(model_data[features].values, model_data["total"].values)

pred_df["LA_2028"] = np.maximum(rf_f.predict(pred_df[features].values),0).round(0).astype(int)
trees_p = np.array([t.predict(pred_df[features].values) for t in rf_f.estimators_])
pred_df["ci_low"]  = np.percentile(trees_p,10,axis=0).clip(0).round(0).astype(int)
pred_df["ci_high"] = np.percentile(trees_p,90,axis=0).round(0).astype(int)
pred_df["Paris_2024"] = [paris_2024[c] for c in pred_df["country"]]
pred_df["is_usa"] = pred_df["country"].apply(lambda c: "USA (host)" if c=="United States" else "Altro")

top15 = pred_df.sort_values("LA_2028", ascending=False).head(15)

# Bar chart previsioni con errori
bars_pred = (
    alt.Chart(top15)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("LA_2028:Q", title="Medaglie previste LA 2028"),
        y=alt.Y("country:N", sort="-x", title=None),
        color=alt.Color("is_usa:N",
            scale=alt.Scale(domain=["USA (host)","Altro"], range=[RED,BLUE]),
            legend=alt.Legend(title="")),
        tooltip=["country:N",
                 alt.Tooltip("Paris_2024:Q", title="Paris 2024 (reale)"),
                 alt.Tooltip("LA_2028:Q", title="LA 2028 (prev.)"),
                 alt.Tooltip("ci_low:Q", title="CI 10%"),
                 alt.Tooltip("ci_high:Q", title="CI 90%")]
    )
)
# Punti Paris 2024 per confronto
dots_paris = (
    alt.Chart(top15)
    .mark_point(color=GRAY, size=60, shape="diamond", opacity=0.7)
    .encode(x="Paris_2024:Q", y=alt.Y("country:N", sort="-x"),
            tooltip=["country:N", alt.Tooltip("Paris_2024:Q", title="Paris 2024")])
)
# Error bars CI
errors_pred = (
    alt.Chart(top15)
    .mark_errorbar(color=NAVY, ticks=True)
    .encode(
        x=alt.X("ci_low:Q", title=""),
        x2="ci_high:Q",
        y=alt.Y("country:N", sort="-x")
    )
)
text_pred = (
    alt.Chart(top15)
    .mark_text(dx=6, fontSize=10, fontWeight="bold")
    .encode(x="LA_2028:Q", y=alt.Y("country:N", sort="-x"),
            text="LA_2028:Q")
)

(bars_pred + errors_pred + dots_paris + text_pred).properties(
    title="Previsioni Los Angeles 2028 — barre di errore = IC 10-90%  ◆ = Paris 2024",
    width=580, height=420
).display()

usa = pred_df[pred_df["country"]=="United States"].iloc[0]
print(f"USA (paese ospitante): {usa['LA_2028']} medaglie previste")
print(f"Paris 2024: {paris_2024['United States']} medaglie")
print(f"Effetto host stimato: +{usa['LA_2028']-paris_2024['United States']} medaglie")


alt.LayerChart(...)

USA (paese ospitante): 107 medaglie previste
Paris 2024: 126 medaglie
Effetto host stimato: +-19 medaglie


## 8. Sintesi finale — i tre notebook

### Il percorso

| Notebook | Domanda | Risultato chiave |
|---|---|---|
| NB1 | Il PIL spiega le medaglie? | r=0.68 — sì, ma è solo un proxy |
| NB2 | La spesa élite spiega le medaglie? | r=0.85 SPLISS, r=0.92 UK Sport |
| NB3 | Possiamo prevedere e isolare la causalità? | R²=0.83 out-of-sample, DiD +17 GB |

### Risposta alla domanda di ricerca

**Sì — gli investimenti in infrastrutture sportive producono medaglie.**

Il segnale è coerente su più livelli (cross-nazione, longitudinale, per disciplina)
e in più paesi (GB, Francia, Australia). Il meccanismo è documentato:
investimento → strutture e atleti → medaglie nel ciclo successivo (lag 4-8 anni).
